# Cài đặt thư viện

In [1]:
!pip install pandas kafka-python pyspark==3.5.0 findspark tensorflow


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip


# Import thư viện

In [2]:
import findspark
from pyspark.sql import SparkSession
import os
import pickle
import tensorflow as tf
import numpy as np
import pandas as pd
from pyspark.sql.types import StructType, IntegerType, StructField, StringType, TimestampType, DoubleType
from pyspark.sql.functions import col, from_json, current_timestamp, hour, dayofweek, to_timestamp, pandas_udf
from tensorflow.keras.models import load_model

# Init spark

In [3]:
findspark.init()

# Cấu hình Spark với Kafka
scala_version = '2.12'
spark_version = '3.5.0'

packages = [
    f'org.apache.spark:spark-sql-kafka-0-10_{scala_version}:{spark_version}',
    'org.apache.kafka:kafka-clients:3.5.0'
]

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("traffic-lstm-prediction") \
    .config("spark.jars.packages", ",".join(packages)) \
    .config("spark.sql.adaptive.enabled", "false") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark Version: {spark.version}")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/hotien/.ivy2/cache
The jars for the packages stored in: /Users/hotien/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
org.apache.kafka#kafka-clients added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-32461c4f-16e7-49db-9c7f-bdd5a8af8c63;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.0 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
	found org.apache.kafka#kafka-clients;3.5.0 in central
	found com.github.luben#zstd-jni;1.5.5-1 in cent

Spark Version: 3.5.0


# Schema cho dữ liệu từ Kafka

In [4]:
vehicle_details_schema = StructType([
    StructField("car", IntegerType(), True),
    StructField("motorbike", IntegerType(), True),
    StructField("bus", IntegerType(), True),
    StructField("truck", IntegerType(), True)
])

kafka_schema = StructType([
    StructField("timestamp_utc", StringType(), True),
    StructField("location", StringType(), True),
    StructField("total_vehicles", IntegerType(), True),
    StructField("vehicle_details", vehicle_details_schema, True),
    StructField("traffic_status", IntegerType(), True)
])

topic_name = 'traffic_detection_topic'
kafka_server = 'localhost:9092'

# Đọc stream từ Kafka và Parse dữ liệu 


In [5]:
kafka_df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_server) \
    .option("subscribe", topic_name) \
    .option("startingOffsets", "latest") \
    .load()

parsed_df = kafka_df.selectExpr("CAST(value AS STRING) as json") \
    .select(from_json(col("json"), kafka_schema).alias("data")) \
    .select("data.*")

final_df = parsed_df \
    .withColumn("timestamp", to_timestamp(col("timestamp_utc"))) \
    .withColumn("hour", hour(col("timestamp"))) \
    .withColumn("dayofweek", dayofweek(col("timestamp"))) \
    .withColumn("processing_time", current_timestamp())

print("Schema của dữ liệu đã parse:")
final_df.printSchema()

Schema của dữ liệu đã parse:
root
 |-- timestamp_utc: string (nullable = true)
 |-- location: string (nullable = true)
 |-- total_vehicles: integer (nullable = true)
 |-- vehicle_details: struct (nullable = true)
 |    |-- car: integer (nullable = true)
 |    |-- motorbike: integer (nullable = true)
 |    |-- bus: integer (nullable = true)
 |    |-- truck: integer (nullable = true)
 |-- traffic_status: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- hour: integer (nullable = true)
 |-- dayofweek: integer (nullable = true)
 |-- processing_time: timestamp (nullable = false)



# Tải model và broadcast các thành phần của model

In [6]:
model_dir = "./models"
model_path = os.path.join(model_dir, "traffic_lstm_model.h5")
feature_columns_path = os.path.join(model_dir, "traffic_features.pkl")
scaler_path = os.path.join(model_dir, "traffic_scaler.pkl")
label_encoder_path = os.path.join(model_dir, "traffic_label_encoder.pkl")

os.makedirs(model_dir, exist_ok=True)

try:
    model = load_model(model_path)
    with open(feature_columns_path, 'rb') as f:
        feature_columns = pickle.load(f)
    with open(scaler_path, 'rb') as f:
        scaler = pickle.load(f)
    with open(label_encoder_path, 'rb') as f:
        label_encoder = pickle.load(f)

    print("Đã tải thành công model và các thành phần")

except Exception as e:
    print(f"Lỗi khi tải model: {e}")
    print("Tạo model mẫu để test...")

    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense
    from sklearn.preprocessing import StandardScaler, LabelEncoder

    model = Sequential([
        LSTM(50, activation='relu', input_shape=(1, 6)),
        Dense(25, activation='relu'),
        Dense(5, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    feature_columns = ['car', 'motorbike', 'bus', 'truck', 'hour', 'dayofweek']
    scaler = StandardScaler()
    label_encoder = LabelEncoder()
    label_encoder.fit(['RẤT_THƯA', 'THƯA', 'BÌNH_THƯỜNG', 'ĐÔNG', 'RẤT_ĐÔNG'])

feature_columns_bc = spark.sparkContext.broadcast(feature_columns)
scaler_bc = spark.sparkContext.broadcast(scaler)
model_bc = spark.sparkContext.broadcast(model)
label_encoder_bc = spark.sparkContext.broadcast(label_encoder)

/Users/hotien/Documents/cao học/HK5/chuyên đề dự đoán/pj/traffic-forecast/.venv/lib/python3.11/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.0 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/hotien/Documents/cao học/HK5/chuyên đề dự đoán/pj/traffic-forecast/.venv/lib/python3.11/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.7.0 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Đã tải thành công model và các thành phần


# UDF Dự Đoán Tình Trạng Giao Thông
UDF xử lý dự đoán tình trạng giao thông sử dụng mô hình LSTM, chuẩn hóa dữ liệu đầu vào và đảm bảo trả về kết quả kiểu string.

In [7]:
@pandas_udf(StringType())
def predict_traffic_udf(car: pd.Series, motorbike: pd.Series, bus: pd.Series,
                        truck: pd.Series, hour: pd.Series, dayofweek: pd.Series) -> pd.Series:
    try:
        # Tạo DataFrame từ các features
        features_df = pd.DataFrame({
            "car": car,
            "motorbike": motorbike,
            "bus": bus,
            "truck": truck,
            "hour": hour,
            "dayofweek": dayofweek
        })

        # Lấy các thành phần từ broadcast
        feature_columns = feature_columns_bc.value
        scaler = scaler_bc.value
        model = model_bc.value
        label_encoder = label_encoder_bc.value

        # Chuẩn hóa dữ liệu
        features_scaled = scaler.transform(features_df[feature_columns])

        # Reshape cho LSTM (batch_size, sequence_length=1, num_features)
        features_reshaped = features_scaled.reshape(-1, 1, len(feature_columns))

        # Dự đoán
        predictions = model.predict(features_reshaped, verbose=0)
        predicted_classes = np.argmax(predictions, axis=1)

        predicted_labels = label_encoder.inverse_transform(predicted_classes)

        predicted_labels = [str(x) for x in predicted_labels]

        return pd.Series(predicted_labels)

    except Exception as e:
        print(f"Lỗi trong dự đoán: {e}")
        return pd.Series(["LỖI"] * len(car))

# Áp dụng UDF cho dự đoán 

In [8]:
df_pred = final_df.withColumn(
    "predicted_traffic_status",
    predict_traffic_udf(
        col("vehicle_details.car"),
        col("vehicle_details.motorbike"),
        col("vehicle_details.bus"),
        col("vehicle_details.truck"),
        col("hour"),
        col("dayofweek")
    )
)

# Chọn các cột để hiển thị
output_df = df_pred.select(
    "timestamp",
    "location",
    "vehicle_details",
    "traffic_status",
    "predicted_traffic_status",
    "processing_time"
)

# Streaming dự báo

In [ ]:
query = output_df.writeStream \
    .format("console") \
    .outputMode("append") \
    .option("truncate", "false") \
    .trigger(processingTime='30 seconds') \
    .start()

print("Bắt đầu xử lý streaming...")
query.awaitTermination()

Bắt đầu xử lý streaming...
-------------------------------------------
Batch: 0
-------------------------------------------
+---------+--------+---------------+--------------+------------------------+---------------+
|timestamp|location|vehicle_details|traffic_status|predicted_traffic_status|processing_time|
+---------+--------+---------------+--------------+------------------------+---------------+
+---------+--------+---------------+--------------+------------------------+---------------+

